# Sistema de Recomendação de Filmes
**FBC (TF-IDF + RRF)  →  Motor Híbrido Cascade  →  FC (SVD)**

In [17]:
import import_ipynb
import engines

CBFEngine    = engines.CBFEngine
CFEngine     = engines.CFEngine
HybridEngine = engines.HybridEngine


## Inicialização dos Engines

In [18]:
cbf    = CBFEngine()
cf     = CFEngine()
hybrid = HybridEngine(cbf_engine=cbf, cf_engine=cf)
print("Engines prontos.")


Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
vetor latente usuário: (610, 100)
vetor latente filme: (8536, 100)
vetor de vies de cada usuário: (610,)
vetor de vies de cada filme: (8536,)
Engines prontos.


## Demo — Recomendações para um Usuário

In [19]:
import pandas as pd

usuario_id = 23
threshold  = 4.0

df_train = pd.read_csv('data_spliting_hybrid/training.csv')
df_test  = pd.read_csv('data_spliting_hybrid/testing.csv')

perfil = df_train[
    (df_train['userId'] == usuario_id) & (df_train['rating'] >= threshold)
][['movieId', 'rating']].copy()

titles = cbf.df_cbf[['movie_id', 'title']].rename(columns={'movie_id': 'movieId'})
perfil = perfil.merge(titles, on='movieId', how='left')
perfil = perfil.sort_values('rating', ascending=False).reset_index(drop=True)
perfil.index = range(1, len(perfil) + 1)

print(f'Perfil do Usuário {usuario_id} — {len(perfil)} filmes bem avaliados no treino (nota >= {threshold}):')
print()
display(perfil[['title', 'rating']].rename(columns={'title': 'Filme', 'rating': 'Nota'}))


Perfil do Usuário 23 — 35 filmes bem avaliados no treino (nota >= 4.0):



,Filme,Nota
1,Full Metal Jacket (1987),5.0
2,"Samouraï, Le (Godson, The) (1967)",5.0
3,Blade Runner (1982),5.0
4,Barton Fink (1991),5.0
5,"Shining, The (1980)",5.0
6,"Third Man, The (1949)",5.0
7,Requiem for a Dream (2000),5.0
8,"Triplets of Belleville, The (Les triplettes de...",4.5
9,Chinatown (1974),4.5
10,Hard-Boiled (Lat sau san taam) (1992),4.5


In [20]:
rec = hybrid.recomendar(usuario_id, k_cbf=200, k=10, alpha=0.2)

# Filmes relevantes no teste (gabarito)
relevantes = set(
    df_test[(df_test['userId'] == usuario_id) & (df_test['rating'] >= threshold)]['movieId']
)

# Marca hits
rec['Hit'] = rec['movie_id'].apply(lambda x: 'HIT' if x in relevantes else '')
rec.index = range(1, len(rec) + 1)

# Métricas
hits      = rec['Hit'].eq('HIT').sum()
precision = hits / len(rec)
recall    = hits / len(relevantes) if relevantes else 0

print(f'Top 10 recomendações para o Usuário {usuario_id}:')
print()
display(rec[['title', 'score_hibrido', 'Hit']].rename(columns={
    'title': 'Filme',
    'score_hibrido': 'Score Híbrido',
    'Hit': 'Relevante?'
}))
print()
print(f'Filmes relevantes no teste (nota >= {threshold}): {len(relevantes)}')
print(f'Precision@10 : {precision:.2%}  ({hits} acertos em 10 recomendações)')
print(f'Recall@10    : {recall:.2%}  ({hits} de {len(relevantes)} filmes relevantes recuperados)')


Top 10 recomendações para o Usuário 23:



,Filme,Score Híbrido,Relevante?
1,Fight Club (1999),0.995171,
2,Seven (a.k.a. Se7en) (1995),0.984254,
3,Memento (2000),0.983432,HIT
4,"Departed, The (2006)",0.981418,
5,2001: A Space Odyssey (1968),0.977850,HIT
6,"Matrix, The (1999)",0.975430,
7,Kiss Kiss Bang Bang (2005),0.973356,
8,Sin City (2005),0.970302,
9,Jackie Brown (1997),0.969893,
10,"Maltese Falcon, The (1941)",0.966886,HIT



Filmes relevantes no teste (nota >= 4.0): 15
Precision@10 : 30.00%  (3 acertos em 10 recomendações)
Recall@10    : 20.00%  (3 de 15 filmes relevantes recuperados)
